In [1]:
import sys, os
from pathlib import Path

ROOT = Path(r"C:\Users\user\doc-asistant\doc-asistant")
sys.path.append(str(ROOT))
sys.path.append(str(ROOT / "notebooks" / "from_src"))

import torch
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("faiss:", faiss.__version__)
print("ROOT:", ROOT)

torch: 2.6.0+cu124
cuda: True
faiss: 1.15.1
ROOT: C:\Users\user\doc-asistant\doc-asistant


In [13]:
# функция чанкинга: 500 символов, перекрытие 50
def chunk_text(text, size=500, overlap=50):
    text = str(text).strip()
    if not text:
        return []
    chunks = []
    start = 0
    step = size - overlap
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step
    return chunks

In [3]:
CSV = ROOT / "data" / "cooking_clean.csv"
df = pd.read_csv(CSV, sep=";")
print("recipes:", len(df))

df["full_text"] = (
    "Recipe: " + df["title"].astype(str) + "\n" +
    "Instructions: " + df["instructions"].astype(str)
)

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

chunks = []
chunk_meta = []

for idx, row in df.iterrows():
    recipe_chunks = chunk_text(row["full_text"], size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    for j, c in enumerate(recipe_chunks):
        chunks.append(c)
        chunk_meta.append({
            "recipe_number": int(row["number"]),
            "recipe_title": str(row["title"]),
            "chunk_index": j,
        })

print("total chunks:", len(chunks))
print("avg chunks/recipe:", round(len(chunks) / len(df), 2))

recipes: 500
total chunks: 1053
avg chunks/recipe: 2.11


In [4]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = SentenceTransformer(MODEL_NAME, device=device)
print("model:", MODEL_NAME)
print("device:", encoder.device)
print("embedding dim:", encoder.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
device: cuda:0
embedding dim: 384


In [5]:
print("encoding", len(chunks), "chunks...")
embeddings = encoder.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # для L2 тоже полезно
)
print("embeddings shape:", embeddings.shape)

encoding 1053 chunks...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

embeddings shape: (1053, 384)


In [6]:
# Индекс по методичке: L2 (евклидово расстояние)
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype("float32"))

print("index total vectors:", index.ntotal)
print("index dim:", index.d)
print("index type: IndexFlatL2")

index total vectors: 1053
index dim: 384
index type: IndexFlatL2


In [7]:
import json

INDEX_PATH = ROOT / "models" / "faiss_recipes_l2.index"
META_PATH = ROOT / "models" / "faiss_recipes_l2_meta.json"

faiss.write_index(index, str(INDEX_PATH))
print("saved index:", INDEX_PATH, INDEX_PATH.stat().st_size, "bytes")

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "encoder": MODEL_NAME,
        "index_type": "IndexFlatL2",
        "metric": "l2",
        "num_chunks": len(chunks),
        "meta": chunk_meta,
    }, f, ensure_ascii=False, indent=2)
print("saved meta:", META_PATH, META_PATH.stat().st_size, "bytes")

saved index: C:\Users\user\doc-asistant\doc-asistant\models\faiss_recipes_l2.index 1617453 bytes
saved meta: C:\Users\user\doc-asistant\doc-asistant\models\faiss_recipes_l2_meta.json 122083 bytes


In [8]:
# поиск top-k по L2: меньше dist = ближе
def search(query, k=10):
    q_emb = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_emb, k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "distance": float(dist),
            "idx": int(idx),
            "meta": chunk_meta[idx],
            "text": chunks[idx],
        })
    return results

test_queries = ["chocolate cake", "шоколадный торт", "apple pie", "рецепт с яблоками"]

for q in test_queries:
    print(f"\nQuery: {q}")
    for r in search(q, k=10):
        print(f"  dist={r['distance']:.3f}  recipe='{r['meta']['recipe_title']}'")


Query: chocolate cake
  dist=0.577  recipe='ultimate chocolate cake'
  dist=0.687  recipe='betsys chocolate fudge cake'
  dist=0.711  recipe='chocolate eclair cake'
  dist=0.732  recipe='chocolate mochadot cake'
  dist=0.742  recipe='chocolate log'
  dist=0.825  recipe='chocolate eclair dessert'
  dist=0.853  recipe='hoho cakedeep dark chocolate hoho cake'
  dist=0.856  recipe='chocolate bread pudding'
  dist=0.857  recipe='golden treasure cake'
  dist=0.860  recipe='sour cream pound cake'

Query: шоколадный торт
  dist=0.552  recipe='ultimate chocolate cake'
  dist=0.668  recipe='betsys chocolate fudge cake'
  dist=0.699  recipe='chocolate eclair cake'
  dist=0.702  recipe='chocolate mochadot cake'
  dist=0.723  recipe='chocolate log'
  dist=0.815  recipe='chocolate eclair dessert'
  dist=0.820  recipe='sour cream pound cake'
  dist=0.825  recipe='hoho cakedeep dark chocolate hoho cake'
  dist=0.834  recipe='golden treasure cake'
  dist=0.856  recipe='strawberry tunnel cream cake'

Q

In [9]:
# контекст из top-10 чанков
def build_context(query, k=10):
    results = search(query, k=k)
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(f"[{i}] Recipe: {r['meta']['recipe_title']}\n{r['text']}")
    return "\n\n".join(context_parts), results

In [10]:
sys.path.append(str(ROOT / "notebooks" / "from_src"))
import llm_client
from openai import OpenAI

client = OpenAI(
    base_url=llm_client.OLLAMA_BASE_URL,
    api_key=llm_client.OLLAMA_API_KEY,
)
print("Ollama ready, model:", llm_client.OLLAMA_MODEL)

Ollama ready, model: gpt-oss:120b


In [11]:
# Жёсткая инструкция
RAG_SYSTEM = """Ты помощник по рецептам. Отвечай кратко, только фактами из контекста.
Если ответа нет в контексте - пиши 'Нет данных'. Не выдумывай."""

def rag_answer(question, k=10):
    context, results = build_context(question, k=k)
    user_prompt = f"Контекст:\n{context}\n\nВопрос: {question}\n\nОтветь кратко на основе контекста."
    
    response = client.chat.completions.create(
        model=llm_client.OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
    )
    answer = response.choices[0].message.content
    sources = [r["meta"]["recipe_title"] for r in results]
    return answer, sources

In [12]:
def llm_no_rag(question):
    response = client.chat.completions.create(
        model=llm_client.OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

q = "Расскажи про рецепт шоколадного торта"
print(f"Q: {q}\n")
print("Без RAG:", llm_no_rag(q))
print("\nС RAG (k=10):")
answer, sources = rag_answer(q)
print(answer)
print("\nSources (10):")
for s in sources:
    print(f" - {s}")

Q: Расскажи про рецепт шоколадного торта

Без RAG: Нет данных

С RAG (k=10):
**Рецепт «Ultimate Chocolate Cake»**  

**Ингредиенты**  
- 3 oz (≈85 г) несладкого шоколада  
- 1 c (≈225 г) несоленого сливочного масла  
- 2 ⅓ c (≈460 г) плотно утрамбованного светло-коричневого сахара  
- 2 ⅓ c (≈300 г) просеянной муки для торта  
- 2/3 c (≈160 мл) пахты  
- 1 c (≈240 мл) кипятка  
- 1 tsp (≈5 мл) ванилина  
- 3 крупных яйца  
- 2 tsp (≈10 мл) пищевой соды  
- 1 tsp (≈5 мл) соли  

**Краткая инструкция**  
1. Разогреть духовку до 350 °F (≈175 °C). Смажьте формы растопленным маслом и присыпьте их просеянной мукой, стряхнув излишки.  
2. Растопите шоколад на водяной бане, затем дайте ему остыть.  
3. В большой миске взбейте сливочное масло с коричневым сахаром до лёгкой и воздушной массы. Поочерёдно добавляйте яйца, хорошо взбивая после каждого.  
4. В отдельной миске смешайте сухие ингредиенты (мука, сода, соль).  
5. Поочерёдно вводите сухие ингредиенты, пахту и остывший шоколад, чередуя и